# 02 Behavior model on the processed dataset (Eating / Locomotion / Resting)

Run top to bottom on a **GPU runtime** (Runtime, Change runtime type, T4 GPU).
Needs `cmz_processed_dataset.zip` in `MyDrive/CMZ/`.

1. get the code and the data
2. make contact sheets, check the labels, list bad stretches in `review_edits.csv`
3. second detection pass for frames where the civet was missed
4. train, evaluate, and write per-second tables

In [ ]:
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/Siwali/cmz-project.git"
BRANCH = "feat/civet-zoo-rules-cvat"
REPO = Path("/content/cmz-project")
PIPE = REPO / "pipeline"

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout + r.stderr)
    return r.stdout + r.stderr

print(git("-C", str(REPO), "pull", REPO_URL, BRANCH) if REPO.exists() else git("clone", "-b", BRANCH, REPO_URL, str(REPO)))

In [ ]:
DRIVE_CMZ = Path("/content/drive/MyDrive/CMZ")
DATA_ZIP = DRIVE_CMZ / "cmz_processed_dataset.zip"
BASE = Path("/content/CMZ")          # local disk: much faster than reading 9,000 files from Drive
EDITS = DRIVE_CMZ / "review_edits.csv"
RUN_DIR = DRIVE_CMZ / "behavior_run1"

if not (BASE / "metadata" / "manifest.csv").exists():
    !unzip -q -n "{DATA_ZIP}" -d "{BASE}"
!ls "{BASE}" "{BASE}/metadata"

## Check the labels
Each clip's label comes from its folder name. A clip can contain a few seconds that don't match it
(for example the civet walks away at the end of a "Resting" clip). The sheets show one picture per second.

In [ ]:
!python "{PIPE}/make_review_sheets.py" --base "{BASE}" --out "{DRIVE_CMZ}/review_sheets"

In [ ]:
# Look at one page. Change the file name to look at other videos (E01_p1.jpg, L03_p1.jpg, R04_p2.jpg ...).
from IPython.display import Image, display
display(Image(filename=str(DRIVE_CMZ / "review_sheets" / "R01_p1.jpg")))

Write the bad stretches into `MyDrive/CMZ/review_edits.csv` (open it from Drive, or edit it here):

```
video_id,start_s,end_s,action
R03,40,55,drop
E02,0,8,relabel:Locomotion
```
`drop` removes the seconds from training. `relabel:<Class>` changes the label. The time is the
second shown under each thumbnail. Keep the header line. An empty file is fine.

In [ ]:
if not EDITS.exists():
    EDITS.write_text("video_id,start_s,end_s,action\n")
print(EDITS.read_text())

## Second detection pass
The first detector missed the civet in most night clips. This re-runs `MDV6-apa-rtdetr-e` on those frames.

In [ ]:
%pip install -q PytorchWildlife

In [ ]:
# The pass takes about 1.5 h on a CPU, so the result is kept on Drive and reused after a restart.
saved = DRIVE_CMZ / "redetect.csv"
if saved.exists():
    !cp "{saved}" "{BASE}/metadata/redetect.csv"
else:
    !python "{PIPE}/redetect.py" --base "{BASE}" --conf 0.15
    !cp "{BASE}/metadata/redetect.csv" "{saved}"
!ls -l "{BASE}/metadata/redetect.csv"

## Train and evaluate

In [ ]:
!python "{PIPE}/train_behavior.py" --base "{BASE}" --edits "{EDITS}" --out "{RUN_DIR}" --epochs 8 --redetect-min-conf 0.35

**Steadier score:** `--cv` tests every video once (5 folds) instead of relying on 3 test videos, and
`--freeze-backbone` trains only the last layer. Try both:

In [ ]:
!python "{PIPE}/train_behavior.py" --base "{BASE}" --edits "{EDITS}" --out "{DRIVE_CMZ}/behavior_cv" --cv --epochs 4 --freeze-backbone --redetect-min-conf 0.35

In [ ]:
import json
import pandas as pd
metrics = json.loads((RUN_DIR / "metrics.json").read_text())
for k, v in metrics.items():
    if isinstance(v, dict) and "accuracy" in v:
        print(f"{k:24s} accuracy {v['accuracy']:.3f}   macro-F1 {v['macro_f1']:.3f}   rows {v['rows']}")
print(pd.read_csv(RUN_DIR / "per_video.csv").to_string(index=False))

Read it like this: `test_model` is the number that matters. `test_shortcut_baseline` is a guess that never
looks at the civet (infrared means Locomotion, a still picture means Resting, otherwise Eating).
If the model is not clearly above the baseline on test, or only good on one lighting condition, it has not learned the behavior yet.

**Add motion.** A single frame cannot show movement, but movement is the main sign of Active vs Inactive.
This combines the CV predictions above with frame-to-frame motion and scores every video with a model that never saw it:

In [ ]:
!python "{PIPE}/stack_motion.py" --base "{BASE}" --pred "{DRIVE_CMZ}/behavior_cv/predictions.csv" --out "{DRIVE_CMZ}/behavior_cv"

## Per-second table and the Active/Inactive sheet

In [ ]:
import pandas as pd
sec = pd.read_csv(RUN_DIR / "per_second.csv")
sec[sec.video_id == sec.video_id.iloc[0]].head(30)

In [ ]:
# Active/Inactive bouts for one test video, using the rule layer (3 s minimum because the clips are short)
vid = sec.video_id.iloc[0]
!python "{PIPE}/civet_rules.py" "{RUN_DIR}/rules/{vid}.csv" --out "{RUN_DIR}/sheets/{vid}" --min-bout 3